In [0]:
dbutils.library.restartPython()

1. Create two (2) new tables in your own databse where you'll store the predictions from each model for this exercise.

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
-- Create tables in own database
USE gr5069.gbm2118;

In [0]:
%sql
-- Table for storing predictions of positionOrder from results dataset (Model 1)
CREATE TABLE results_ridge_reg (
alt INT,
lng FLOAT,
lat FLOAT,
actual_speed DOUBLE,
predicted_speed DOUBLE,
residual DOUBLE
);

In [0]:
%sql
-- Table for storing predictions of point scored from results dataset (Model 2)
CREATE TABLE results_random_forest (
grid INT,
points_sprint FLOAT,
actual_points FLOAT,
predicted_points FLOAT,
residual FLOAT
);

2. Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments

## Model 1: Ridge Regression

I will predict the average fastest lap speed in km/h using ridge regression. My features are: circuit altitude in meters (alt), circuit longitude (lng) and circuit latitude (lat). I will first convert the target and features, and raceId to ints and floats, depending on their original datatypes, as shown on [mintfly](https://www.mintlify.com/TracingInsights/RaceData/reference/results). I will also convert the fastest lap speed to meters per second so that it is consistent with the circuit altitude.

In [0]:
# Import important libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark
import mlflow.sklearn

In [0]:
# Load the results data
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
df_circuits = spark.read.csv("/Volumes/gr5069/raw/f1_data/circuits.csv", header=True)

# Inspect the data
display(df_results)
display(df_races)
display(df_circuits)

In [0]:
# Replace malformed values with NULLs before changing datatype
from pyspark.sql.functions import col, when, trim

# df_results
df_results = df_results.withColumn(
    "fastestLapSpeed",
    when(trim(col("fastestLapSpeed")) == "\\N", None)
    .otherwise(col("fastestLapSpeed"))
)

# df_circuits
df_circuits = df_circuits.withColumn(
    "lat",
    when(trim(col("lat")) == "\\N", None)
    .otherwise(col("lat"))
)

df_circuits = df_circuits.withColumn(
    "lng",
    when(trim(col("lng")) == "\\N", None)
    .otherwise(col("lng"))
)

df_circuits = df_circuits.withColumn(
    "alt",
    when(trim(col("alt")) == "\\N", None)
    .otherwise(col("alt"))
)

In [0]:
# Convert raceId in df_results to int, and fastestLapSpeed to float 
df_results = df_results.withColumn("fastestLapSpeed", df_results["fastestLapSpeed"].cast("float"))
df_results = df_results.withColumn("raceId", df_results["raceId"].cast("int"))

# Convert raceId and circuitId in df_races to integer
df_races = df_races.withColumn("raceId", df_races["raceId"].cast("int"))
df_races = df_races.withColumn("circuitId", df_races["circuitId"].cast("int"))

# Convert circuitId and alt in df_circuits to integer and lat and lng to floats
df_circuits = df_circuits.withColumn("circuitId", df_circuits["circuitId"].cast("int"))
df_circuits = df_circuits.withColumn("lat", df_circuits["lat"].cast("float"))
df_circuits = df_circuits.withColumn("lng", df_circuits["lng"].cast("float"))
df_circuits = df_circuits.withColumn("alt", df_circuits["alt"].cast("int"))

# Inspect the data
display(df_results)
display(df_races)
display(df_circuits)

I will calculate the average fastestLapSpeed per circuitId. Before that I will merge the results and races dataframes and then merge the resulting dataframe with the circuits dataframe. This is because we are interested in how the circuit features predict fastestLapSpeed, so everything has to be at circuit level. I will also convert fastestLapSpeed to m/s to be consistent with the meter unit of altitude. 

After calculating the average fastestLapSpeed in m/s, I remerge the df_avg dataframe with the original dataset that it was derived from and keep only the important features. I also drop duplicates and nulls for some light cleaning.

In [0]:
# merge df_results and df_races on raceId
df = df_results.join(df_races, on="raceId", how="inner")

# merge df and df_circuits on circuitId
df = df.join(df_circuits, on="circuitId", how="inner")
# Inspect the data
display(df)

In [0]:
# Convert fastestLapSpeed to m/s (1m/s = 3.6km/h)
df = df.withColumn("fastestLapSpeed", df["fastestLapSpeed"] / 3.6)
display(df)

In [0]:
# Calculate average fastestLapSpeed for each circuitId
df_avg = df.groupBy("circuitId").agg({"fastestLapSpeed": "avg"})
display(df_avg)

In [0]:
#rename avg(fastestLapSpeed) avg_fastestLapSpeed
df_avg = df_avg.withColumnRenamed("avg(fastestLapSpeed)", "avg_fastestLapSpeed")

# Step 2: join back, keeping unique rows
df_grouped = df_avg.join(df, on="circuitId", how="left")

# Step 3: select columns
df_grouped = df_grouped.select(
    "circuitId",
    "raceId",
    "avg_fastestLapSpeed",
    "alt",
    "lng",
    "lat"
)

# Drop duplicates
df_grouped = df_grouped.drop_duplicates(["circuitId", "raceId"])

# Drop Null Values
df_grouped = df_grouped.dropna()

display(df_grouped)

Do a train/test split 

In [0]:
# Change variable names to X, y to prepare for train/test split
df_grouped = df_grouped.toPandas()
y = df_grouped["avg_fastestLapSpeed"]
X = df_grouped[["alt", "lng", "lat"]]

# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Build a ridge regression model

In [0]:
# Ridge Regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error

params = {"alpha": 1.0}
rr = Ridge(**params)
rr.fit(X_train, y_train.dropna())
predictions = rr.predict(X_test)

# Create metrics to evaluate model
mse = mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
mae_med = median_absolute_error(y_test, predictions)

print("  mse: {}".format(mse))
print("  mae: {}".format(mae))
print("  R2: {}".format(r2))
print("  mae_med: {}".format(mae_med))

# Display coeffiecients
coefficients = pd.DataFrame(rr.coef_, index=X_train.columns, columns=["coefficients"])
coefficients.sort_values("coefficients", ascending=False)
print(coefficients)

# Create plot
fig, ax = plt.subplots()

sns.residplot(x=predictions, y=y_test.astype(float))
plt.xlabel("Predicted values for average fastestLapSpeed in (m/s)")
plt.ylabel("Residual")
plt.title("Residual Plot")

In [0]:
import mlflow.sklearn
import tempfile
import os

with mlflow.start_run(run_name="Basic Ridge Experiment") as run:
  
  # Log model
  mlflow.sklearn.log_model(rr, "ridge-regression")

  # Log params
  [mlflow.log_param(param, value) for param, value in params.items()]
  
  # Log metrics
  mlflow.log_metric("mse", mse)
  mlflow.log_metric("mae", mae)  
  mlflow.log_metric("r2", r2)
  mlflow.log_metric("mae_med", mae_med)
  
  # Log coefficients using a temporary file
  temp = tempfile.NamedTemporaryFile(prefix="ridge-coefficients-", suffix=".csv")
  temp_name = temp.name
  try:
      coefficients.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "ridge-coefficients.csv")

  finally:
      temp.close() # Delete the temp file
  
  # Log residuals using a temporary file
  temp = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")
  temp_name = temp.name
  try:
    fig.savefig(temp_name)
    mlflow.log_artifact(temp_name, "residuals.png")
  finally:
    temp.close() # Delete the temp file
    
  display(fig)
  
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

![image_1777412720783.png](./image_1777412720783.png "image_1777412720783.png")

## Model 2: Random Forest

I will predict the points scored by each driver per race in the results dataframe using random forest regression. My features are starting grid position (grid) from the results dataset and points scored in the sprint race (points) from the sprint_results dataset. I will first load both datasets into dataframes then rename the "points" column in sprint_results to points_sprint to avoid confusion with the points column in the results dataset. I will then convert the points and points_sprint to float and grid, driverId and raceId to integer data types. After which, I will merge both dataframes on driverId and raceId.

In [0]:
# Load results dataset
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

#Load sprint_results dataset
sprint_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/sprint_results.csv", header=True)

# Inspect data
display(results)
display(sprint_results)

In [0]:
# rename points column in sprint_results to points_sprint
sprint_results = sprint_results.withColumnRenamed("points", "points_sprint")

# Inspect the data
display(sprint_results)

In [0]:
# Replace malformed values with NULLs before changing datatypes
from pyspark.sql.functions import col, when, trim

# results
results = results.withColumn(
    "points",
    when(trim(col("points")) == "\\N", None)
    .otherwise(col("points"))
)

# sprint_points
sprint_results = sprint_results.withColumn(
    "points_sprint",
    when(trim(col("points_sprint")) == "\\N", None)
    .otherwise(col("points_sprint"))
)

In [0]:
# Convert raceId, driverId, points and grid in results to integers
results = results.withColumn("raceId", results["raceId"].cast("int"))
results = results.withColumn("driverId", results["driverId"].cast("int"))
results = results.withColumn("points", results["points"].cast("float"))
results = results.withColumn("grid", results["grid"].cast("int"))

# Convert raceId, driverId and points_sprint in sprint_results to integers
sprint_results = sprint_results.withColumn("raceId", sprint_results["raceId"].cast("int"))
sprint_results = sprint_results.withColumn("driverId", sprint_results["driverId"].cast("int"))
sprint_results = sprint_results.withColumn("points_sprint", sprint_results["points_sprint"].cast("float"))

# Inspect the data
display(results)
display(sprint_results)

In [0]:
# Merge results and sprint_results on raceId and driverId
results = results.join(sprint_results, on=["raceId", "driverId"], how="inner")

# Inspect the data
display(results)

Do a Train/Test Split

In [0]:
# Change variable names to X, y to prepare for train/test split
df2 = results.toPandas()
y2 = df2["points"]
X2 = df2[["grid", "points_sprint"]]

# Train/test split
from sklearn.model_selection import train_test_split

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

Build a Random Forest regression model

In [0]:
# Random Forest Regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error

params = {"n_estimators": 200, "max_depth": 5, "random_state": 42}
rf = RandomForestRegressor(**params)
rf.fit(X2_train, y2_train)
predictions2 = rf.predict(X2_test)

# Create metrics to evaluate model
mse_2 = mean_squared_error(y2_test, predictions2)
mae_2 = mean_absolute_error(y2_test, predictions2)
r2_2 = r2_score(y2_test, predictions)
mae_med_2 = median_absolute_error(y2_test, predictions2)
print("  mse: {}".format(mse_2))
print("  mae: {}".format(mae_2))
print("  R2: {}".format(r2_2))
print("  mae_med: {}".format(mae_med_2))

# Create feature importance
feature_importances = pd.DataFrame(rf.feature_importances_, index=X2_train.columns, columns=["importance"])
feature_importances.sort_values("importance", ascending=False)
print(feature_importances)

# Create plot
fig, ax = plt.subplots()

sns.residplot(x=predictions2, y=y2_test.astype(float))
plt.xlabel("Predicted values for Driver Points")
plt.ylabel("Residual")
plt.title("Residual Plot")

In [0]:
import mlflow.sklearn
import tempfile
import os

with mlflow.start_run(run_name="Basic RF Experiment") as run:
  
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-model")

  # Log params
  [mlflow.log_param(param, value) for param, value in params.items()]
  
  # Log metrics
  mlflow.log_metric("mse", mse_2)
  mlflow.log_metric("mae", mae_2)  
  mlflow.log_metric("r2", r2_2)  
  mlflow.log_metric("mae_med", mae_med_2)
  
  # Log importances using a temporary file
  temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
  temp_name = temp.name
  try:
      feature_importances.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")

  finally:
      temp.close() # Delete the temp file
  
  # Log residuals using a temporary file
  temp = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")
  temp_name = temp.name
  try:
    fig.savefig(temp_name)
    mlflow.log_artifact(temp_name, "residuals.png")
  finally:
    temp.close() # Delete the temp file
    
  display(fig)
  
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

3. For each model, store its predictions in the corresponding table you created in your own database. Ensure you are using your own database to store your predictions.

## Model 1

In [0]:
# Create a results dataframe
results_df = pd.DataFrame({
    "actual_speed": y_test.values,
    "predicted_speed": predictions
})

# Residuals
results_df["residual"] = results_df["actual_speed"] - results_df["predicted_speed"]

In [0]:
# Recover identifiers
results_df["alt"] = X_test["alt"].values
results_df["lng"] = X_test["lng"].values
results_df["lat"] = X_test["lat"].values

display(results_df)

In [0]:
#Convert pandas to Spark
spark_df = spark.createDataFrame(results_df)

In [0]:
# Write to Databricks table
spark_df.write.mode("append").saveAsTable("gr5069.gbm2118.results_ridge_reg")

## Model 2

In [0]:
# Create a results dataframe
results_df2 = pd.DataFrame({
    "actual_points": y2_test.values,
    "predicted_points": predictions2
})

# Residuals
results_df2["residual"] = results_df2["actual_speed"] - results_df2["predicted_speed"]

In [0]:
#  Recover identifiers
results_df2["grid"] = X2_test["grid"].values
results_df2["points_sprint"] = X_test["points_sprint"].values

In [0]:
#Convert pandas to Spark
spark_df2 = spark.createDataFrame(results_df2)

In [0]:
# Write to Databricks table
spark_df2.write.mode("append").saveAsTable("gr5069.gbm2118.results_random_forest")